# Module 1: Sponsor Profiling & Typology

**Question:** Who are the AI policy actors, and how do they differ?

Analyses:
1. Degree distributions — primary sponsorship vs. cosponsorship counts
2. Policy breadth vs. depth — specialists vs. generalists
3. Chamber asymmetry — Senate vs. House patterns
4. Taxonomy fingerprints — cluster sponsors by AGORA annotation profiles

In [39]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import defaultdict

import analysis_utils as au

docs_df = au.load_comprehensive_df()
cosp_df = au.load_cosponsors_df()
communities = au.load_communities()

print(f"Documents: {len(docs_df)}")
print(f"Cosponsor records: {len(cosp_df)}")
print(f"Unique cosponsors: {cosp_df['Cosponsor_BioguideId'].nunique()}")
print(f"Communities: {len(communities)}")

Documents: 581
Cosponsor records: 3462
Unique cosponsors: 521
Communities: 3


## 1.1 Degree Distributions

Build the bipartite sponsor↔document graph, then measure:
- **Cosponsorship degree**: how many documents each sponsor co-sponsors
- **Document degree**: how many cosponsors each document attracts
- Compare primary sponsors (from `Party_Code` in docs) vs. cosponsors

In [3]:
# Build bipartite graph
B = au.build_sponsor_document_bigraph(docs_df, cosp_df)

# Separate node sets
sponsor_nodes = {n for n, d in B.nodes(data=True) if d.get("bipartite") == 1}
doc_nodes = {n for n, d in B.nodes(data=True) if d.get("bipartite") == 0}

# Degree per sponsor (number of documents cosponsored)
sponsor_degrees = pd.DataFrame([
    {"bioguide": n.replace("sponsor:", ""),
     "name": B.nodes[n].get("name", ""),
     "party": B.nodes[n].get("party", ""),
     "state": B.nodes[n].get("state", ""),
     "doc_count": B.degree(n)}
    for n in sponsor_nodes
]).sort_values("doc_count", ascending=False)

# Degree per document (number of cosponsors)
doc_degrees = pd.DataFrame([
    {"agora_id": n.replace("doc:", ""),
     "name": B.nodes[n].get("name", "")[:80],
     "policy_area": B.nodes[n].get("policy_area", ""),
     "cosponsor_count": B.degree(n)}
    for n in doc_nodes
]).sort_values("cosponsor_count", ascending=False)

print(f"Sponsors in graph: {len(sponsor_degrees)}")
print(f"Documents in graph: {len(doc_degrees)}")
print(f"\nTop 10 most active cosponsors:")
sponsor_degrees.head(10)

Sponsors in graph: 521
Documents in graph: 581

Top 10 most active cosponsors:


,bioguide,name,party,state,doc_count
271,D000624,"Rep. Dingell, Debbie [D-MI-6]",D,MI,39
424,R000603,"Rep. Rouzer, David [R-NC-7]",R,NC,37
279,L000599,"Rep. Lawler, Michael [R-NY-17]",R,NY,37
19,S000344,"Rep. Sherman, Brad [D-CA-32]",D,CA,36
69,N000179,"Rep. Napolitano, Grace F. [D-CA-32]",D,CA,36
54,G000551,"Rep. Grijalva, Raúl M. [D-AZ-7]",D,AZ,36
354,S001218,"Rep. Stansbury, Melanie Ann [D-NM-1]",D,NM,36
316,W000826,"Rep. Wild, Susan [D-PA-7]",D,PA,34
455,M001219,"Del. Moylan, James C. [R-GU-At Large]",R,GU,32
329,T000460,"Rep. Thompson, Mike [D-CA-4]",D,CA,32


In [31]:
# Sponsor degree distribution (log-scale histogram)
fig_sponsor_deg = px.histogram(
    sponsor_degrees, x="doc_count", color="party",
    color_discrete_map={"D": "#2166ac", "R": "#b2182b", "I": "#4daf4a", "": "#999999"},
    nbins=40, barmode="group", opacity=1,
    labels={"doc_count": "Documents Co-sponsored", "party": "Party"},
    title="Sponsor Degree Distribution: How Many Bills Does Each Cosponsor Sign?",
)
fig_sponsor_deg.update_layout(yaxis_type="log", yaxis_title="Count (log scale)",
                               template="plotly_white", height=450)
fig_sponsor_deg.show()

# Document degree distribution
fig_doc_deg = px.histogram(
    doc_degrees[doc_degrees["cosponsor_count"] <25],
    x="cosponsor_count", color="policy_area",
    nbins=10, barmode="stack",
    labels={"cosponsor_count": "Number of Cosponsors", "policy_area": "Policy Area"},
    title="Document Degree Distribution: How Many Cosponsors Per Bill?",
)
fig_doc_deg.update_layout(template="plotly_white", height=450)
fig_doc_deg.show()

## 1.2 Policy Breadth vs. Depth

For each cosponsor, count:
- **Breadth**: number of distinct policy areas they cosponsor in
- **Depth**: total number of documents they cosponsor

Sponsors in the upper-right are **generalist high-volume** actors. Lower-right are **specialist high-volume** (many bills, few areas). Upper-left are **generalist low-volume** (few bills, surprisingly diverse).

In [32]:
# Merge cosponsor records with document policy areas
cosp_with_policy = cosp_df.merge(
    docs_df[["AGORA ID", "Policy_Area"]], on="AGORA ID", how="left"
)

# Breadth and depth per sponsor
breadth_depth = cosp_with_policy.groupby("Cosponsor_BioguideId").agg(
    depth=("AGORA ID", "nunique"),
    breadth=("Policy_Area", "nunique"),
    party=("Cosponsor_Party", "first"),
    state=("Cosponsor_State", "first"),
    name=("Cosponsor_FullName", "first"),
).reset_index()

fig_bd = px.scatter(
    breadth_depth, x="depth", y="breadth",
    color="party",
    color_discrete_map={"D": "#2166ac", "R": "#b2182b", "I": "#4daf4a"},
    hover_data=["name", "state"],
    labels={"depth": "Documents Co-sponsored (Depth)",
            "breadth": "Distinct Policy Areas (Breadth)",
            "party": "Party"},
    title="Sponsor Typology: Specialists vs. Generalists",
    opacity=0.7,
)
fig_bd.update_layout(template="plotly_white", height=500)
fig_bd.show()

# Tag the extremes
top_generalists = breadth_depth.nlargest(10, "breadth")
top_specialists = breadth_depth[breadth_depth["breadth"] <= 2].nlargest(10, "depth")
print("Top 10 Generalists (most policy areas):")
print(top_generalists[["name", "party", "state", "depth", "breadth"]].to_string(index=False))
print("\nTop 10 Specialists (≤2 areas, most bills):")
print(top_specialists[["name", "party", "state", "depth", "breadth"]].to_string(index=False))

Top 10 Generalists (most policy areas):
                                       name party state  depth  breadth
             Rep. Lawler, Michael [R-NY-17]     R    NY     37       13
Del. Norton, Eleanor Holmes [D-DC-At Large]     D    DC     29       12
        Rep. Fitzpatrick, Brian K. [R-PA-1]     R    PA     23       11
            Rep. Bonamici, Suzanne [D-OR-1]     D    OR     28       10
                Rep. Carson, Andre [D-IN-7]     D    IN     18       10
              Rep. Dingell, Debbie [D-MI-6]     D    MI     39       10
             Rep. Gottheimer, Josh [D-NJ-5]     D    NJ     19       10
               Sen. Heinrich, Martin [D-NM]     D    NM     21       10
       Rep. Stansbury, Melanie Ann [D-NM-1]     D    NM     36       10
              Rep. Salinas, Andrea [D-OR-6]     D    OR     12       10

Top 10 Specialists (≤2 areas, most bills):
                               name party state  depth  breadth
          Rep. Graves, Sam [R-MO-6]     R    MO     31      

## 1.3 Chamber Asymmetry

Compare Senate vs. House:
- Average cosponsors per bill
- Policy area mix
- Cosponsor party composition

In [33]:
# Infer chamber from cosponsor full name (Sen. vs Rep.)
cosp_df["chamber"] = cosp_df["Cosponsor_FullName"].apply(
    lambda x: "Senate" if str(x).startswith("Sen.") else
              ("House" if str(x).startswith("Rep.") else "Unknown")
)

# Per-document stats by chamber of cosponsors
doc_chamber = cosp_df.groupby(["AGORA ID", "chamber"]).size().unstack(fill_value=0)
doc_chamber = doc_chamber.reset_index().merge(
    docs_df[["AGORA ID", "Policy_Area"]], on="AGORA ID", how="left"
)

# Average cosponsors per bill by chamber
chamber_avg = pd.DataFrame({
    "Senate cosponsors (avg)": [doc_chamber.get("Senate", pd.Series([0])).mean()],
    "House cosponsors (avg)": [doc_chamber.get("House", pd.Series([0])).mean()],
})
print("Average cosponsors per document by chamber:")
print(chamber_avg.to_string(index=False))

# Policy area breakdown by chamber
chamber_policy = cosp_df.merge(
    docs_df[["AGORA ID", "Policy_Area"]], on="AGORA ID", how="left"
)
chamber_policy_counts = chamber_policy.groupby(["chamber", "Policy_Area"]).size().reset_index(name="count")
top_areas = chamber_policy_counts.groupby("Policy_Area")["count"].sum().nlargest(10).index
chamber_policy_top = chamber_policy_counts[chamber_policy_counts["Policy_Area"].isin(top_areas)]

fig_chamber = px.bar(
    chamber_policy_top, x="Policy_Area", y="count", color="chamber",
    barmode="group",
    color_discrete_map={"Senate": "#1b7837", "House": "#762a83"},
    labels={"count": "Cosponsor-Document Pairs", "Policy_Area": "Policy Area"},
    title="Chamber Asymmetry: Senate vs. House Cosponsorship by Policy Area",
)
fig_chamber.update_layout(template="plotly_white", height=450,
                           xaxis_tickangle=-35)
fig_chamber.show()

Average cosponsors per document by chamber:
 Senate cosponsors (avg)  House cosponsors (avg)
                1.558087                6.111617


## 1.4 Taxonomy Fingerprints

Aggregate each sponsor's AGORA taxonomy exposure (Strategies, Harms, Applications, Risk Factors) into a profile vector, then cluster sponsors by similarity. This goes beyond policy area — it captures *how* sponsors approach AI regulation.

In [59]:
au.sponsor_taxonomy_profile(docs_df, cosponsors_df, group="Strategies")

NameError: name 'cosponsors_df' is not defined

In [ ]:
# Build taxonomy profiles for each sponsor across all groups
profiles = {}
for group in ["Strategies", "Harms", "Applications", "Risk Factors", "Incentives"]:
    prof = au.sponsor_taxonomy_profile(docs_df, cosp_df, group)
    prof.columns = [f"{group}: {c}" for c in prof.columns]
    print(prof.columns)
    profiles[group] = prof



print(profiles["Strategies"])
# Combine into a single fingerprint matrix

fingerprint = pd.concat(profiles.values(), axis=1).fillna(0)
print(f"Fingerprint matrix: {fingerprint.shape[0]} sponsors × {fingerprint.shape[1]} taxonomy tags")

# Normalize per-sponsor (row-normalize to proportions)
fingerprint_norm = fingerprint.div(fingerprint.sum(axis=1), axis=0).fillna(0)

# Top 30 most active sponsors for heatmap readability
top30 = sponsor_degrees.head(30)["bioguide"].values
fp_top30 = fingerprint_norm.loc[fingerprint_norm.index.isin(top30)]

# Add names for display
name_map = dict(zip(cosp_df["Cosponsor_BioguideId"], cosp_df["Cosponsor_FullName"]))
fp_top30.index = [name_map.get(b, b)[:35] for b in fp_top30.index]

# Heatmap
fig_fp = px.imshow(
    fp_top30.values,
    x=[c.split(": ", 1)[-1][:30] for c in fp_top30.columns],
    y=fp_top30.index.tolist(),
    color_continuous_scale="YlOrRd",
    labels={"color": "Proportion"},
    title="Taxonomy Fingerprints: Top 30 Most Active Cosponsors",
    aspect="auto",
)
fig_fp.update_layout(template="plotly_white", height=700, width=1100,
                      xaxis_tickangle=-45, xaxis_side="bottom")
fig_fp.show()

Index(['Strategies: AGORA ID', 'Strategies: Convening',
       'Strategies: Disclosure', 'Strategies: Disclosure: About evaluation',
       'Strategies: Disclosure: About incidents',
       'Strategies: Disclosure: About inputs',
       'Strategies: Disclosure: Accuracy thereof',
       'Strategies: Disclosure: In deployment',
       'Strategies: Disclosure: In standard form', 'Strategies: Evaluation',
       'Strategies: Evaluation: Adversarial testing',
       'Strategies: Evaluation: Conformity assessment',
       'Strategies: Evaluation: External auditing',
       'Strategies: Evaluation: Impact assessment',
       'Strategies: Evaluation: Post-market monitoring',
       'Strategies: Governance development',
       'Strategies: Government study or report',
       'Strategies: Government support',
       'Strategies: Government support: AI workforce-related',
       'Strategies: Government support: For R&D', 'Strategies: Input controls',
       'Strategies: Input controls: Compute c

In [ ]:
# Cluster sponsors by taxonomy fingerprint using graph-based approach:
# Build a sponsor-sponsor similarity graph (cosine similarity on fingerprints),
# then detect communities.

# Use all sponsors with >= 3 documents for meaningful fingerprints
active_mask = fingerprint.sum(axis=1) >= 3
fp_active = fingerprint_norm.loc[active_mask]

# Cosine similarity matrix using NumPy
fp_array = fp_active.values
norms = np.linalg.norm(fp_array, axis=1, keepdims=True)
fp_normalized = fp_array / norms
sim_matrix = fp_normalized @ fp_normalized.T
np.fill_diagonal(sim_matrix, 0)

# Build similarity graph (threshold at median similarity for sparsity)
threshold = np.median(sim_matrix[sim_matrix > 0])
G_sim = nx.Graph()
bios = fp_active.index.tolist()
for i in range(len(bios)):
    G_sim.add_node(bios[i], name=name_map.get(bios[i], bios[i]),
                   party=cosp_df.loc[cosp_df["Cosponsor_BioguideId"] == bios[i], "Cosponsor_Party"].iloc[0]
                   if bios[i] in cosp_df["Cosponsor_BioguideId"].values else "")
    for j in range(i + 1, len(bios)):
        if sim_matrix[i, j] > threshold:
            G_sim.add_edge(bios[i], bios[j], weight=float(sim_matrix[i, j]))

# Community detection on similarity graph
from networkx.algorithms.community import greedy_modularity_communities
tax_communities = list(greedy_modularity_communities(G_sim, weight="weight"))
print(f"Taxonomy-based communities: {len(tax_communities)}")
for i, comm in enumerate(tax_communities):
    parties = [G_sim.nodes[n].get("party", "") for n in comm]
    d_count = parties.count("D")
    r_count = parties.count("R")
    print(f"  Community {i}: {len(comm)} members, {d_count}D / {r_count}R")

## 1.5 Exports

Save key outputs for downstream modules and dashboard.

In [ ]:
# Export sponsor profiles
au.save_df(sponsor_degrees, "sponsor_degrees")
au.save_df(breadth_depth, "sponsor_breadth_depth")
au.save_df(fingerprint_norm, "sponsor_taxonomy_fingerprints")
au.save_graph(G_sim, "sponsor_similarity_graph")

# Save interactive figures as HTML for dashboard
fig_sponsor_deg.write_html(str(au.OUTPUTS_DIR / "fig_sponsor_degree_dist.html"))
fig_bd.write_html(str(au.OUTPUTS_DIR / "fig_breadth_depth.html"))
fig_chamber.write_html(str(au.OUTPUTS_DIR / "fig_chamber_asymmetry.html"))
fig_fp.write_html(str(au.OUTPUTS_DIR / "fig_taxonomy_fingerprints.html"))

print("Exports saved to notebooks/outputs/")